Today's objectives
1. Extract information from datetime columns.
2. Create new features from existing columns.
3. Apply conditional transformations.
4. Understand map(), apply() and vectorized operations.
5. Handle categorical variables.
6. Prepare a dataset for machine learning.

| Method | Typical use |
|---|---|
| `map()` | Replace values using a dictionary or function |
| `apply()` | Apply a custom function to values, rows or columns |
| `np.where()` | Apply a simple condition efficiently |

apply() is flexible, but for simple numerical conditions, vectorized operations or np.where() are usually faster.

In [4]:
import pandas as pd
import numpy as np
df=pd.read_csv("df_no_missing.csv")
df["OrderDate"]=pd.to_datetime(df["OrderDate"])

In [5]:
df.head()

,OrderID,Product,Category,Quantity,UnitPrice,Region,OrderDate,Quantity_imputed,Revenue,OrderType,CustomerID
0,1001,Laptop,Electronics,2,800,North,2026-09-01,False,1600,High,201
1,1002,Mouse,Electronics,5,20,South,2026-09-02,False,100,Standard,202
2,1003,Keyboard,Electronics,3,50,North,2026-09-03,False,150,Standard,203
3,1004,Laptop,Electronics,1,800,East,2026-09-04,False,800,Standard,201
4,1005,Monitor,Electronics,2,250,West,2026-09-05,True,500,Standard,204


Down we are going to extract year, month, and day 

In [ ]:
df["Year"]=df["OrderDate"].dt.year
df["Month"]=df["OrderDate"].dt.month
df["DayOfWeek"]=df["OrderDate"].dt.day_name()

In [7]:
df.head()

,OrderID,Product,Category,Quantity,UnitPrice,Region,OrderDate,Quantity_imputed,Revenue,OrderType,CustomerID,Year,Month,DayOfWeek
0,1001,Laptop,Electronics,2,800,North,2026-09-01,False,1600,High,201,2026,9,Tuesday
1,1002,Mouse,Electronics,5,20,South,2026-09-02,False,100,Standard,202,2026,9,Wednesday
2,1003,Keyboard,Electronics,3,50,North,2026-09-03,False,150,Standard,203,2026,9,Thursday
3,1004,Laptop,Electronics,1,800,East,2026-09-04,False,800,Standard,201,2026,9,Friday
4,1005,Monitor,Electronics,2,250,West,2026-09-05,True,500,Standard,204,2026,9,Saturday


In [10]:
#Count the number of orders for each weekday.
df.groupby(df["DayOfWeek"])["OrderID"].count().sort_values(ascending=False)

DayOfWeek
Friday       2
Saturday     2
Thursday     2
Tuesday      2
Wednesday    2
Monday       1
Sunday       1
Name: OrderID, dtype: int64

In [16]:
df.groupby(df["DayOfWeek"])["OrderID"].count().sort_values(ascending=False).idxmax()

'Friday'

In [20]:
counts=df.groupby("DayOfWeek")["OrderID"].count()
busiest_day=counts[counts==counts.max()]
print(busiest_day)  # Names and counts
busiest_day.index    # Only weekday names

DayOfWeek
Friday       2
Saturday     2
Thursday     2
Tuesday      2
Wednesday    2
Name: OrderID, dtype: int64


Index(['Friday', 'Saturday', 'Thursday', 'Tuesday', 'Wednesday'], dtype='object', name='DayOfWeek')

In [23]:
df["CategoryCode"]=df["Category"].map({"Electronics": "E",
    "Furniture": "F"})
df["CategoryCode"].isna().sum()

0

In [26]:
df["PriceLevel"]=np.where(df["UnitPrice"]>=300, "High", "Low")
df.head(2)

,OrderID,Product,Category,Quantity,UnitPrice,Region,OrderDate,Quantity_imputed,Revenue,OrderType,CustomerID,Year,Month,DayOfWeek,CategoryCode,PriceLevel
0,1001,Laptop,Electronics,2,800,North,2026-09-01,False,1600,High,201,2026,9,Tuesday,E,High
1,1002,Mouse,Electronics,5,20,South,2026-09-02,False,100,Standard,202,2026,9,Wednesday,E,Low


map() is designed for element-wise mapping of a Series. It is commonly used with dictionaries or functions.

apply() is more flexible. It can work on a Series element by element, or on DataFrame rows and columns.

In [31]:
def q_level(x):
    if x <= 2:
        return "Low"
    else:
        return "High"
df["QuantityLevel"]=df["Quantity"].apply(q_level)
df[["Product", "Quantity", "QuantityLevel"]]
df["Quantity"].apply(q_level)
df["Quantity"].map(q_level)

0      Low
1     High
2     High
3      Low
4      Low
5     High
6      Low
7      Low
8     High
9      Low
10    High
11     Low
Name: Quantity, dtype: object

In [ ]:
#df["QuantityLevel"] = df["Quantity"].apply(
 #   lambda x: "Low" if x <= 2 else "High")

In [34]:
df.groupby(["Category"]).agg(
    averga_price=("UnitPrice","mean"),
    averga_quanity=("Quantity","mean"),
    total_rev=("Revenue","sum"))

,averga_price,averga_quanity,total_rev
Category,,,
Electronics,306.0,2.9,6330
Furniture,300.0,1.5,900


In [36]:
df[["Product", "Category", "CategoryCode", "UnitPrice", "PriceLevel", "Quantity", "QuantityLevel"]]

,Product,Category,CategoryCode,UnitPrice,PriceLevel,Quantity,QuantityLevel
0,Laptop,Electronics,E,800,High,2,Low
1,Mouse,Electronics,E,20,Low,5,High
2,Keyboard,Electronics,E,50,Low,3,High
3,Laptop,Electronics,E,800,High,1,Low
4,Monitor,Electronics,E,250,Low,2,Low
5,Mouse,Electronics,E,20,Low,4,High
6,Keyboard,Electronics,E,50,Low,2,Low
7,Desk,Furniture,F,300,High,1,Low
8,Laptop,Electronics,E,800,High,3,High
9,Monitor,Electronics,E,250,Low,2,Low
